<a href="https://colab.research.google.com/github/S-V-Kartheek/multimodal-graph-recommender/blob/main/MM_CLightRec_ML1M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MM-CLightRec: Contrastive Multimodal LightGCN (MovieLens 1M Dataset)

This notebook trains the MM-CLightRec model on the **MovieLens 1M** dataset using a T4 GPU. The architecture includes:
1. **Unified LightGCN** (replaces GCN+GAT)
2. **Modality-Specific Channels** with adaptive fusion
3. **Inter-Modal Contrastive Loss ($L_1$)**
4. **Structural Contrastive Loss ($L_2$)**
5. **Cold-Start Contrastive Loss ($L_3$)**

### Step 1: Upload the Project
Upload the `mm_clightrec_ml1m_code.zip` file to the root of this Colab session using the folder icon on the left panel, then run the cell below to extract it.

In [3]:
!unzip -o -q mm_clightrec_ml1m_code.zip

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Install Required Libraries
This installs PyTorch, PyTorch Geometric, and Transformers/SentenceTransformers.

In [4]:
!pip install torch torchvision torchaudio sentence-transformers scikit-learn pandas numpy
!pip install transformers
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 119.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 137.0 MB/s eta 0:00:00


### Step 3: Run Training on MovieLens 1M
This will extract text/image/video proxies, process metadata, build the unified bipartite graph, and train the MM-CLightRec architecture for the target 300 epochs.

In [5]:
!python main.py --dataset ml1m --epochs 300 --batch_size 4096 --include_cold_start

/content/models/vgae.py:24: SyntaxWarning: invalid escape sequence '\h'
  \hat{A} = D^{-1/2} A D^{-1/2}
  MM-CLightRec: Contrastive Multimodal LightGCN Recommendation
  Dataset: ml1m
L3 Cold-Start Active  : Yes
  Version: JOURNAL (with L3 cold-start)

[STEP 1] Loading and preprocessing ml1m dataset...
[INFO] Downloading MovieLens 1M dataset...
[INFO] Download complete.
[INFO] Extracting dataset...
[INFO] Extraction complete.
[INFO] Loaded 1000209 ratings, 6040 users, 3883 movies
[INFO] Users: 6040, Items: 3883
[INFO] Encoding user features...
[INFO] Encoding multimodal item features...
    - Extracting Text Data using TF-IDF + SVD...
    - Generating Image Data (synthetic proxy)...
    - Generating Video Data (synthetic proxy)...
    - Processing Metadata (One-Hot Encoding & Normalization)...
[INFO] Building bipartite graph...
[INFO] Split: train=641373, val=80133, test=82596
[INFO] Data loading completed in 12.8 seconds
[INFO] Users: 4832, Items: 3883
[INFO] User features: torch.Size(

### Step 4: Zip Results for Download
Zip the results folder to easily view loss plots and metrics locally.

In [6]:
!zip -r results_ml1m.zip results/

  adding: results/ (stored 0%)
  adding: results/metrics_over_epochs_ml1m.png (deflated 6%)
  adding: results/training_loss_ml1m.png (deflated 13%)
  adding: results/checkpoint_ml1m_epoch5.pth (deflated 8%)
  adding: results/link_prediction_ml1m.png (deflated 24%)
  adding: results/checkpoint_ml1m_epoch65.pth (deflated 7%)
  adding: results/checkpoint_ml1m_epoch55.pth (deflated 7%)
  adding: results/checkpoint_ml1m_epoch105.pth (deflated 7%)
  adding: results/loss_components_ml1m.png (deflated 9%)
  adding: results/checkpoint_ml1m_epoch75.pth (deflated 7%)
  adding: results/checkpoint_ml1m_epoch15.pth (deflated 8%)
  adding: results/checkpoint_ml1m_epoch40.pth (deflated 8%)
  adding: results/metrics_ml1m.png (deflated 22%)
  adding: results/checkpoint_ml1m_epoch25.pth (deflated 8%)
  adding: results/checkpoint_ml1m_epoch30.pth (deflated 8%)
  adding: results/checkpoint_ml1m_epoch50.pth (deflated 7%)
  adding: results/checkpoint_ml1m_epoch85.pth (deflated 7%)
  adding: results/checkpoin

In [17]:
# ════════════════════════════════════════════════════════════════════════
# SAMPLED NEGATIVE EVALUATION  (matches base paper protocol)
# No retraining needed — loads best saved model
# Protocol: 1 positive + 99 random negatives per user → rank 100 items
# ════════════════════════════════════════════════════════════════════════

import torch, numpy as np, os

N_NEGATIVE = 99   # Standard: 99 negatives + 1 positive = 100 candidates
K = 10            # Top-K

# ── Load best saved model ──────────────────────────────────────────────
BEST_MODEL_PATH = 'results/best_model.pth'
if not os.path.exists(BEST_MODEL_PATH):
    BEST_MODEL_PATH = 'results/mm_clightrec_ml1m.pth'

print(f'[INFO] Loading model from {BEST_MODEL_PATH}...')
state = torch.load(BEST_MODEL_PATH, map_location=device)
# Handle both formats (raw state_dict or full checkpoint)
if 'model_state_dict' in state:
    model.load_state_dict(state['model_state_dict'])
else:
    model.load_state_dict(state)
model.eval()
print('[INFO] Model loaded ✅')

# ── Get all model scores once (n_users × n_items matrix) ──────────────
print('[INFO] Computing full score matrix...')
with torch.no_grad():
    score_matrix = model.get_all_scores(
        user_features, item_features, edge_index,
        user_modality_features, item_modality_features
    ).cpu().numpy()  # shape: (n_users, n_items)
print(f'[INFO] Score matrix: {score_matrix.shape}')

# ── Sampled Negative Evaluation ────────────────────────────────────────
def compute_ndcg_sampled(ranked_list, pos_item, k):
    for rank, item in enumerate(ranked_list[:k]):
        if item == pos_item:
            return 1.0 / np.log2(rank + 2)
    return 0.0

def evaluate_sampled(score_matrix, test_user_items, train_user_items,
                     n_items, n_neg=99, k=10, seed=42):
    rng = np.random.RandomState(seed)
    prec_list, rec_list, ndcg_list, f1_list = [], [], [], []

    for u, pos_items in test_user_items.items():
        if not pos_items:
            continue

        # Use the FIRST positive test item (standard protocol)
        pos_item = pos_items[0]

        # All items the user has interacted with (train + other test) — exclude as negatives
        seen = set(train_user_items.get(u, [])) | set(pos_items)

        # Sample 99 random negatives the user has NOT seen
        all_items = np.arange(n_items)
        unseen    = np.array([i for i in all_items if i not in seen])
        if len(unseen) < n_neg:
            negs = unseen
        else:
            negs = rng.choice(unseen, size=n_neg, replace=False)

        # Build candidate pool: [pos_item] + 99 negatives
        candidates = np.concatenate([[pos_item], negs])

        # Get model scores for this candidate pool
        s = score_matrix[u][candidates]

        # Rank by score descending
        order     = np.argsort(-s)
        ranked    = candidates[order]

        # ── Metrics ───────────────────────────────────────────────────
        hits = int(pos_item in ranked[:k])
        p    = hits / k
        r    = hits / 1  # only 1 positive in sampled eval
        nd   = compute_ndcg_sampled(ranked, pos_item, k)
        f    = 2*p*r/(p+r) if (p+r) > 0 else 0.0

        prec_list.append(p)
        rec_list.append(r)
        ndcg_list.append(nd)
        f1_list.append(f)

    return {
        f'Precision@{k} (Sampled)': float(np.mean(prec_list)),
        f'Recall@{k} (Sampled)':    float(np.mean(rec_list)),
        f'NDCG@{k} (Sampled)':      float(np.mean(ndcg_list)),
        f'F1@{k} (Sampled)':        float(np.mean(f1_list)),
        'Users evaluated':          len(prec_list),
    }

# ── Run it ─────────────────────────────────────────────────────────────
print(f'\n[INFO] Running Sampled Negative Evaluation (1 pos + {N_NEGATIVE} neg)...')
sampled_metrics = evaluate_sampled(
    score_matrix,
    test_user_items=data['test_user_items'],
    train_user_items=data['train_user_items'],
    n_items=data['n_items'],
    n_neg=N_NEGATIVE, k=K
)

print('\n' + '='*60)
print(f'  SAMPLED EVALUATION  (1 pos + {N_NEGATIVE} neg, K={K})')
print(f'  Protocol matches base paper (MGRS-HFA)')
print('='*60)
for metric, value in sampled_metrics.items():
    if isinstance(value, float):
        print(f'  {metric:<30} {value:.4f}')
    else:
        print(f'  {metric:<30} {value}')
print('='*60)

print('\n── Comparison Table ──')
print(f'{"Metric":<25} {"Base Paper":>12} {"Ours (All-Item)":>16} {"Ours (Sampled)":>15}')
print('-'*70)
p_s  = sampled_metrics[f'Precision@{K} (Sampled)']
r_s  = sampled_metrics[f'Recall@{K} (Sampled)']
nd_s = sampled_metrics[f'NDCG@{K} (Sampled)']
f_s  = sampled_metrics[f'F1@{K} (Sampled)']
print(f'{"Precision@10":<25} {"0.8269":>12} {"0.0442":>16} {p_s:>15.4f}')
print(f'{"Recall@10":<25} {"0.8718":>12} {"0.0502":>16} {r_s:>15.4f}')
print(f'{"NDCG@10":<25} {"0.6844":>12} {"0.0580":>16} {nd_s:>15.4f}')
print(f'{"F1@10":<25} {"0.8484":>12} {"0.0390":>16} {f_s:>15.4f}')


[INFO] Loading model from results/best_model.pth...


NameError: name 'model' is not defined